# 02. Baseline pipeline: построение данных и проверка работоспособности

**Цель раздела.** Построить и обосновать baseline-пайплайн: как из strict contract получается protocol-fixed постановка, какие признаки используются, почему выбран такой split, и подтверждают ли простые модели, что данные содержат сигнал для раннего предупреждения.

Этот ноутбук решает две задачи:

1. Описать построение пайплайна данных и причины проектных решений.
2. Обучить baseline-модели, чтобы проверить работоспособность данных, признаков и протокола оценки до финальной мультимодальной модели.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc, precision_recall_curve, roc_curve

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
CONTRACT_PATH = ARTIFACT_DIR / "contract" / "strict_intensity_contract.csv.gz"
REPORT_DIR = ARTIFACT_DIR / "reports"
BASELINE_DIR = REPORT_DIR / "protocol_fixed_baselines"

def run_command(command: list[str]) -> None:
    print(" ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

def safe_read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def file_status_table(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for name, path in paths.items():
        rows.append({
            "артефакт": name,
            "путь": str(path.relative_to(PROJECT_ROOT)) if path.exists() else str(path),
            "найден": path.exists(),
            "размер_байт": path.stat().st_size if path.exists() and path.is_file() else np.nan,
        })
    return pd.DataFrame(rows)

def normalize_contract(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for col in ["window_ts", "window_end", "event_anchor"]:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], utc=True, errors="coerce")
    if "y_anomaly" not in out.columns and {"y_warning", "y_risk"}.issubset(out.columns):
        out["y_anomaly"] = np.maximum(out["y_warning"].astype(int), out["y_risk"].astype(int))
    return out


## 1. Роль baseline в исследовательском контуре

**Цель раздела.** Зафиксировать, что baseline не заменяет финальную модель. Он проверяет постановку данных: есть ли измеримый сигнал в логах, KPI, признаках доступности и их простом объединении.


In [ ]:
pipeline_steps = pd.DataFrame(
    [
        {"этап": 1, "шаг": "raw_data", "смысл": "исходные логи и KPI без использования старых промежуточных результатов"},
        {"этап": 2, "шаг": "strict contract", "смысл": "единые временные окна, target, split и служебные признаки доступности"},
        {"этап": 3, "шаг": "protocol-fixed split", "смысл": "оценка с учетом workload, validation threshold и test без подбора порога"},
        {"этап": 4, "шаг": "baseline features", "смысл": "логовые, KPI, deviation и availability признаки"},
        {"этап": 5, "шаг": "baseline models", "смысл": "контрольная проверка, что данные и pipeline дают отделимый score"},
    ]
)
display(pipeline_steps)


**Локальный вывод.** Baseline stage нужен как проверка работоспособности данных и протокола. Если простые модели не видят сигнала, усложнение архитектуры не решает основную проблему постановки.


## 2. Strict contract и причины его устройства

**Цель раздела.** Проверить вход baseline: contract должен содержать окна, split, target, доступность модальностей и признаки для простых моделей.


In [ ]:
if not CONTRACT_PATH.exists():
    run_command([sys.executable, "-m", "dashboard.pipeline.build_contract", "--raw-data", "raw_data"])

contract = normalize_contract(safe_read_csv(CONTRACT_PATH))
if contract.empty:
    raise FileNotFoundError(f"Не найден strict contract: {CONTRACT_PATH}")

contract_report = safe_read_json(REPORT_DIR / "final_data_pipeline_report.json")
contract_summary = safe_read_csv(ARTIFACT_DIR / "contract" / "strict_contract_summary.csv")

print(f"Строк в contract: {len(contract):,}")
display(contract_summary if not contract_summary.empty else contract.head(5))
display(contract.groupby("split").agg(rows=("run_id", "size"), runs=("run_id", "nunique"), workloads=("workload", "nunique")).reset_index())


In [ ]:
target_cols = [c for c in ["y_warning", "y_risk", "y_anomaly", "horizon_30s", "horizon_60s", "horizon_120s", "horizon_300s"] if c in contract.columns]
availability_cols = [c for c in ["log_available", "kpi_available", "kpi_series_available", "has_both_modalities", "kpi_avail_ratio"] if c in contract.columns]

target_rate = contract.groupby("split")[target_cols].mean().T * 100 if target_cols else pd.DataFrame()
availability = contract.groupby("split")[availability_cols].mean().T * 100 if availability_cols else pd.DataFrame()

if not target_rate.empty:
    display(target_rate.round(2).rename_axis("target").reset_index())
    ax = target_rate.plot(kind="bar", figsize=(11, 4), title="Доля positive-окон по target и split, %")
    ax.set_xlabel("Target")
    ax.set_ylabel("Positive windows, %")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()

if not availability.empty:
    display(availability.round(2).rename_axis("feature").reset_index())
    ax = availability.plot(kind="bar", figsize=(10, 4), title="Доступность модальностей по split, %")
    ax.set_xlabel("Признак")
    ax.set_ylabel("Окна, %")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()


**Локальный вывод.** Contract задает единый формат окна. Наличие target и признаков доступности позволяет отдельно проверять логи, KPI и служебный сигнал полноты данных.


## 3. Split, workload и защита оценки

**Цель раздела.** Показать, почему baseline оценивается в protocol-fixed постановке: порог выбирается на validation, test остается только для финального измерения качества.


In [ ]:
workload_split = (
    contract.groupby(["workload", "split"])
    .size()
    .rename("rows")
    .reset_index()
)
display(workload_split)

pivot = workload_split.pivot(index="workload", columns="split", values="rows").fillna(0)
fig, ax = plt.subplots(figsize=(10, 4))
pivot.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Окна по workload и split")
ax.set_xlabel("Workload")
ax.set_ylabel("Количество окон")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()


In [ ]:
if target_cols:
    workload_target = (
        contract.groupby("workload")[target_cols]
        .mean()
        .mul(100)
        .round(2)
        .reset_index()
    )
    display(workload_target)


**Локальный вывод.** Workload влияет на распределение окон и target. Поэтому baseline должен проверяться не только в среднем, но и по workload.


## 4. Группы признаков и baseline-модели

**Цель раздела.** Объяснить, какие контрольные модели строятся и какую часть пайплайна каждая из них проверяет.


In [ ]:
baseline_design = pd.DataFrame(
    [
        {"baseline": "availability_only", "что проверяет": "достаточно ли одного факта доступности логов/KPI", "ожидание": "должен быть слабым контрольным уровнем"},
        {"baseline": "log_tfidf", "что проверяет": "есть ли сигнал в текстах логов", "ожидание": "полезен для warning/anomaly"},
        {"baseline": "metric_raw", "что проверяет": "есть ли сигнал в сырых KPI-статистиках", "ожидание": "полезен для длительных горизонтов"},
        {"baseline": "metric_raw_plus_deviation", "что проверяет": "добавляет ли пользу отклонение/динамика KPI", "ожидание": "должен улучшать отделение части событий"},
        {"baseline": "fused_log_metric", "что проверяет": "работает ли простое объединение логов и KPI", "ожидание": "контроль для мультимодального подхода"},
    ]
)
display(baseline_design)


In [ ]:
manifest = safe_read_json(BASELINE_DIR / "baseline_protocol_manifest.json")
feature_sets = manifest.get("feature_sets", {}) if isinstance(manifest, dict) else {}
if feature_sets:
    feature_table = pd.DataFrame(
        [
            {
                "baseline": name,
                "число_numeric_признаков": cfg.get("numeric_columns"),
                "использует_текст": cfg.get("uses_text"),
            }
            for name, cfg in feature_sets.items()
        ]
    )
    display(feature_table)
else:
    print("Manifest baseline еще не сформирован; таблица признаков появится после обучения.")


**Локальный вывод.** Набор baseline покрывает основные источники сигнала: доступность данных, логи, KPI, динамику KPI и простое объединение модальностей.


## 5. Обучение baseline и проверка артефактов

**Цель раздела.** Запустить обучение baseline-моделей и убедиться, что все необходимые артефакты сформированы.


In [ ]:
TRAIN_BASELINES = True
baseline_command = [
    sys.executable, "-m", "dashboard.training.baselines",
    "--contract-path", str(CONTRACT_PATH),
    "--output-dir", str(BASELINE_DIR),
    "--split-protocol", "stratified_by_workload",
    "--targets", "default",
    "--seed", "43",
    "--validation-seed", "43",
    "--fp-budget", "0.10",
]

if TRAIN_BASELINES:
    run_command(baseline_command)
else:
    print("Baseline training skipped; existing artifacts will be read.")

required = {
    "baseline_protocol_metrics.csv": BASELINE_DIR / "baseline_protocol_metrics.csv",
    "baseline_protocol_predictions.csv.gz": BASELINE_DIR / "baseline_protocol_predictions.csv.gz",
    "baseline_protocol_workload_metrics.csv": BASELINE_DIR / "baseline_protocol_workload_metrics.csv",
    "baseline_protocol_manifest.json": BASELINE_DIR / "baseline_protocol_manifest.json",
}
display(file_status_table(required))


**Локальный вывод.** После этого этапа baseline становится воспроизводимой контрольной линией: метрики, предсказания, breakdown по workload и manifest доступны как файлы.


## 6. Window-level метрики baseline

**Цель раздела.** Оценить, какие baseline-варианты лучше всего проверяют работоспособность данных по каждому target.


In [ ]:
metrics = safe_read_csv(BASELINE_DIR / "baseline_protocol_metrics.csv")
if metrics.empty:
    raise FileNotFoundError("Нет baseline_protocol_metrics.csv")

test_best = metrics.query("split == 'test' and threshold_policy == 'validation_best_f1'").copy()
score_cols = ["average_precision", "roc_auc", "precision", "recall", "f1", "false_positives", "missed_anomalies"]
for col in score_cols:
    test_best[col] = pd.to_numeric(test_best[col], errors="coerce")

best_by_target = test_best.sort_values(["target", "f1"], ascending=[True, False]).groupby("target", as_index=False).head(1)
display(best_by_target[["target", "baseline", "precision", "recall", "f1", "average_precision", "roc_auc", "false_positives", "missed_anomalies"]].round(4))


In [ ]:
heat = test_best.pivot_table(index="target", columns="baseline", values="f1", aggfunc="max")
display(heat.round(4))

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(heat.fillna(0).to_numpy(), aspect="auto", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=30, ha="right")
ax.set_yticks(np.arange(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_title("F1 baseline-вариантов по target")
fig.colorbar(im, ax=ax, label="F1")
plt.tight_layout()


In [ ]:
plot_targets = ["horizon_60s", "horizon_120s", "horizon_300s", "y_warning", "y_risk"]
plot_df = best_by_target[best_by_target["target"].isin(plot_targets)].set_index("target")[["precision", "recall", "f1"]]
if not plot_df.empty:
    ax = plot_df.plot(kind="bar", figsize=(10, 4), title="Лучший baseline по target: Precision / Recall / F1")
    ax.set_xlabel("Target")
    ax.set_ylabel("Значение метрики")
    ax.set_ylim(0, 1.0)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()


**Локальный вывод.** Если хотя бы простые baseline-модели стабильно отделяют часть target, значит contract, split и признаки содержат проверяемый сигнал для следующего этапа.


## 7. Score separation и пороги

**Цель раздела.** Проверить не только итоговую F1, но и форму разделения классов по score: распределения, PR-кривые и ROC-кривые.


In [ ]:
pred = safe_read_csv(BASELINE_DIR / "baseline_protocol_predictions.csv.gz")
if pred.empty:
    print("Файл baseline_protocol_predictions.csv.gz не найден.")
else:
    key_targets = ["horizon_60s", "horizon_120s", "horizon_300s"]
    selected_pairs = best_by_target[best_by_target["target"].isin(key_targets)][["target", "baseline"]].drop_duplicates()

    for _, row in selected_pairs.iterrows():
        target = row["target"]
        baseline = row["baseline"]
        g = pred[
            (pred["pred_split"].astype(str) == "test")
            & (pred["target_name"].astype(str) == target)
            & (pred["baseline"].astype(str) == baseline)
        ].copy()
        if g.empty:
            continue
        fig, ax = plt.subplots(figsize=(8, 3.5))
        for label, title in [(0, "normal"), (1, "positive")]:
            values = pd.to_numeric(g.loc[g["target"].astype(int) == label, "score"], errors="coerce").dropna()
            ax.hist(values, bins=30, alpha=0.55, label=title, density=True)
        ax.set_title(f"Score separation: {target} / {baseline}")
        ax.set_xlabel("score")
        ax.set_ylabel("density")
        ax.legend()
        plt.tight_layout()


In [ ]:
if pred.empty:
    print("Нет prediction-файла для PR/ROC.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    plotted = 0
    for _, row in best_by_target[best_by_target["target"].isin(["horizon_60s", "horizon_120s", "horizon_300s"])].iterrows():
        target = row["target"]
        baseline = row["baseline"]
        g = pred[
            (pred["pred_split"].astype(str) == "test")
            & (pred["target_name"].astype(str) == target)
            & (pred["baseline"].astype(str) == baseline)
        ].copy()
        if g.empty:
            continue
        y = pd.to_numeric(g["target"], errors="coerce").fillna(0).astype(int).to_numpy()
        s = pd.to_numeric(g["score"], errors="coerce").fillna(0).to_numpy()
        if len(np.unique(y)) < 2:
            continue
        pr, rc, _ = precision_recall_curve(y, s)
        fpr, tpr, _ = roc_curve(y, s)
        axes[0].plot(rc, pr, label=f"{target} / {baseline}")
        axes[1].plot(fpr, tpr, label=f"{target} / {baseline} (AUC={auc(fpr, tpr):.3f})")
        plotted += 1

    if plotted:
        axes[0].set_title("PR-кривые лучших baseline")
        axes[0].set_xlabel("Recall")
        axes[0].set_ylabel("Precision")
        axes[0].legend(fontsize=8)
        axes[1].set_title("ROC-кривые лучших baseline")
        axes[1].set_xlabel("False Positive Rate")
        axes[1].set_ylabel("True Positive Rate")
        axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1)
        axes[1].legend(fontsize=8)
        plt.tight_layout()
    else:
        print("Недостаточно данных для PR/ROC.")


**Локальный вывод.** Score separation показывает, насколько pipeline дает ранжирование окон, а не только одну точку качества после выбора порога.


## 8. FP-budget и workload breakdown

**Цель раздела.** Проверить более тихий режим порога и посмотреть, не скрывает ли средняя метрика слабые workload.


In [ ]:
fp_view = metrics.query("split == 'test' and threshold_policy.str.contains('fp_budget', regex=False)", engine="python").copy()
fp_best = fp_view.sort_values(["target", "f1"], ascending=[True, False]).groupby("target", as_index=False).head(1)
display(fp_best[["target", "baseline", "threshold_policy", "precision", "recall", "f1", "false_positives", "missed_anomalies"]].round(4))


In [ ]:
workload_metrics = safe_read_csv(BASELINE_DIR / "baseline_protocol_workload_metrics.csv")
if workload_metrics.empty:
    print("baseline_protocol_workload_metrics.csv не найден.")
else:
    best_keys = best_by_target[["target", "baseline"]].drop_duplicates()
    workload_best = workload_metrics.merge(best_keys, on=["target", "baseline"], how="inner")
    display(workload_best[["target", "baseline", "workload", "precision", "recall", "f1", "false_positives", "missed_anomalies"]].round(4))

    focus = workload_best[workload_best["target"].isin(["horizon_60s", "horizon_120s", "horizon_300s"])]
    if not focus.empty:
        pivot = focus.pivot_table(index="workload", columns="target", values="f1", aggfunc="max")
        fig, ax = plt.subplots(figsize=(8, 4))
        im = ax.imshow(pivot.fillna(0).to_numpy(), aspect="auto", vmin=0, vmax=1)
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=25, ha="right")
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_title("F1 лучших baseline по workload")
        fig.colorbar(im, ax=ax, label="F1")
        plt.tight_layout()


**Локальный вывод.** FP-budget показывает продуктовый компромисс между ложными срабатываниями и пропусками, а workload breakdown показывает, где pipeline требует дальнейшего усиления.


## 9. Итоговая проверка пайплайна

**Цель раздела.** Сформулировать, что именно подтвердил baseline stage перед переходом к финальной модели.


In [ ]:
verdict_rows = []
if not best_by_target.empty:
    for target in ["horizon_60s", "horizon_120s", "horizon_300s", "y_warning", "y_risk"]:
        row = best_by_target[best_by_target["target"] == target]
        if row.empty:
            continue
        row = row.iloc[0]
        verdict_rows.append(
            {
                "проверка": target,
                "лучший_baseline": row["baseline"],
                "F1": round(float(row["f1"]), 4),
                "Precision": round(float(row["precision"]), 4),
                "Recall": round(float(row["recall"]), 4),
                "вывод": "сигнал есть" if float(row["f1"]) >= 0.35 else "сигнал слабый",
            }
        )
display(pd.DataFrame(verdict_rows))


**Локальный вывод.** Baseline stage подтверждает, что strict contract, protocol-fixed split и признаки дают проверяемый сигнал. Следующий этап - `03_fused_model.ipynb`, где этот же контур используется для обучения финальной мультимодальной модели и сравнения с baseline.
